In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/open-problems-multimodal/sample_submission.csv
/kaggle/input/competitions/open-problems-multimodal/train_cite_targets.h5
/kaggle/input/competitions/open-problems-multimodal/metadata_cite_day_2_donor_27678.csv
/kaggle/input/competitions/open-problems-multimodal/test_multi_inputs.h5
/kaggle/input/competitions/open-problems-multimodal/evaluation_ids.csv
/kaggle/input/competitions/open-problems-multimodal/train_cite_inputs.h5
/kaggle/input/competitions/open-problems-multimodal/train_multi_targets.h5
/kaggle/input/competitions/open-problems-multimodal/train_multi_inputs.h5
/kaggle/input/competitions/open-problems-multimodal/metadata.csv
/kaggle/input/competitions/open-problems-multimodal/test_cite_inputs_day_2_donor_27678.h5
/kaggle/input/competitions/open-problems-multimodal/test_cite_inputs.h5
/kaggle/input/datasets/huangyufeid/cite-final-artifacts/model_config.json
/kaggle/input/datasets/huangyufeid/cite-final-artifacts/cite_test_predictions_svd128_celltype_rid

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib


def find_unique_file(filename):
    matches = list(
        Path("/kaggle/input").rglob(filename)
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"没有找到：{filename}"
        )

    if len(matches) > 1:
        print(f"{filename} 找到多个版本：")
        for path in matches:
            print(" ", path)

        raise RuntimeError(
            f"请手动指定 {filename} 的正确路径。"
        )

    return matches[0]


MULTIOME_MODEL_DIR = Path(
    "/kaggle/input/datasets/"
    "huangyufeid/multiome-final-model/"
    "multiome_final"
)

MULTIOME_BACKUP_DIR = Path(
    "/kaggle/input/datasets/"
    "huangyufeid/multiome-final-backup/"
    "multiome_final"
)

files = {
    # CITE
    "cite_predictions":
        find_unique_file(
            "cite_test_predictions_svd128_celltype_ridge1000.h5"
        ),

    # Multiome 模型文件
    "target_svd":
        MULTIOME_MODEL_DIR
        / "target_svd256_final.joblib",

    "x_scaler":
        MULTIOME_MODEL_DIR
        / "x_scaler_final.joblib",

    "z_scaler":
        MULTIOME_MODEL_DIR
        / "z_scaler_final.joblib",

    "multi_gene_ids":
        MULTIOME_MODEL_DIR
        / "target_gene_ids.npy",

    # Multiome 最终 test prediction 及其 cell 顺序
    "z_test_ensemble":
        MULTIOME_BACKUP_DIR
        / "Z_test_latent_ensemble.npy",

    "multi_cell_ids":
        MULTIOME_BACKUP_DIR
        / "test_multi_cell_ids.npy",

    # Test input SVD
    "x_test_svd":
        find_unique_file(
            "X_test_multi_svd128.npy"
        ),

    # Competition files
    "evaluation_ids":
        find_unique_file(
            "evaluation_ids.csv"
        ),

    "sample_submission":
        find_unique_file(
            "sample_submission.csv"
        ),
}

In [7]:
print("File check")
print("=" * 100)

missing_files = []

for name, path in files.items():
    if path.is_file():
        size_mb = (
            path.stat().st_size
            / 1024**2
        )

        print(
            f"{name:<22} "
            f"OK       "
            f"{size_mb:>10.2f} MB  "
            f"{path}"
        )
    else:
        print(
            f"{name:<22} "
            f"MISSING  "
            f"{path}"
        )

        missing_files.append(path)

if missing_files:
    raise FileNotFoundError(
        "以下文件不存在：\n"
        + "\n".join(
            map(str, missing_files)
        )
    )

print("\nAll required files were found.")

File check
cite_predictions       OK            21.81 MB  /kaggle/input/datasets/huangyufeid/cite-final-artifacts/cite_test_predictions_svd128_celltype_ridge1000.h5
target_svd             OK            22.87 MB  /kaggle/input/datasets/huangyufeid/multiome-final-model/multiome_final/target_svd256_final.joblib
x_scaler               OK             0.00 MB  /kaggle/input/datasets/huangyufeid/multiome-final-model/multiome_final/x_scaler_final.joblib
z_scaler               OK             0.01 MB  /kaggle/input/datasets/huangyufeid/multiome-final-model/multiome_final/z_scaler_final.joblib
multi_gene_ids         OK             1.34 MB  /kaggle/input/datasets/huangyufeid/multiome-final-model/multiome_final/target_gene_ids.npy
z_test_ensemble        OK            54.62 MB  /kaggle/input/datasets/huangyufeid/multiome-final-backup/multiome_final/Z_test_latent_ensemble.npy
multi_cell_ids         OK             2.56 MB  /kaggle/input/datasets/huangyufeid/multiome-final-backup/multiome_final/test_mu

In [8]:
import numpy as np

Z_test_saved = np.load(
    files["z_test_ensemble"],
    mmap_mode="r"
)

multi_cell_ids = np.load(
    files["multi_cell_ids"],
    allow_pickle=False
).astype(str)

print(
    "Z_test_saved:",
    Z_test_saved.shape,
    Z_test_saved.dtype
)

print(
    "multi_cell_ids:",
    multi_cell_ids.shape,
    multi_cell_ids.dtype
)

assert Z_test_saved.shape == (
    55935,
    256
)

assert multi_cell_ids.shape == (
    55935,
)

assert len(np.unique(multi_cell_ids)) == (
    len(multi_cell_ids)
)

assert np.isfinite(
    np.asarray(
        Z_test_saved[:1000]
    )
).all()

print(
    "\nZ_test and test cell IDs "
    "are structurally consistent."
)

Z_test_saved: (55935, 256) float32
multi_cell_ids: (55935,) <U12

Z_test and test cell IDs are structurally consistent.


In [9]:
import hashlib


def sha256_file(path, block_size=1024**2):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            block = file.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


common_filenames = [
    "target_svd256_final.joblib",
    "x_scaler_final.joblib",
    "z_scaler_final.joblib",
    "target_gene_ids.npy",
]

print("Cross-dataset consistency check")
print("=" * 80)

for filename in common_filenames:
    model_path = (
        MULTIOME_MODEL_DIR / filename
    )

    backup_path = (
        MULTIOME_BACKUP_DIR / filename
    )

    if (
        model_path.is_file()
        and backup_path.is_file()
    ):
        model_hash = sha256_file(
            model_path
        )

        backup_hash = sha256_file(
            backup_path
        )

        identical = (
            model_hash == backup_hash
        )

        print(
            f"{filename:<35} "
            f"{'IDENTICAL' if identical else 'DIFFERENT'}"
        )

        if not identical:
            raise RuntimeError(
                f"两个 Dataset 中的 {filename} "
                "并不相同，暂时不能混用。"
            )
    else:
        print(
            f"{filename:<35} "
            "only available in one dataset"
        )

Cross-dataset consistency check
target_svd256_final.joblib          IDENTICAL
x_scaler_final.joblib               IDENTICAL
z_scaler_final.joblib               IDENTICAL
target_gene_ids.npy                 IDENTICAL


In [10]:
mlp_paths = sorted(
    MULTIOME_MODEL_DIR.glob(
        "final_mlp_seed_*.keras"
    )
)

for path in mlp_paths:
    print(path.name)

assert len(mlp_paths) == 5

final_mlp_seed_123.keras
final_mlp_seed_2026.keras
final_mlp_seed_2027.keras
final_mlp_seed_42.keras
final_mlp_seed_7.keras


Load articrafts

In [11]:
import gc
import io
import math
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.auto import tqdm


# CITE predictions
cite_predictions = pd.read_hdf(
    files["cite_predictions"]
)

# Multiome components
target_svd = joblib.load(
    files["target_svd"]
)

x_scaler = joblib.load(
    files["x_scaler"]
)

z_scaler = joblib.load(
    files["z_scaler"]
)

Z_test_saved = np.load(
    files["z_test_ensemble"],
    mmap_mode="r"
)

X_test_svd = np.load(
    files["x_test_svd"],
    mmap_mode="r"
)

multi_gene_ids = np.load(
    files["multi_gene_ids"],
    allow_pickle=False
).astype(str)

multi_cell_ids = np.load(
    files["multi_cell_ids"],
    allow_pickle=False
).astype(str)


print("CITE predictions:", cite_predictions.shape)
print("Multiome input SVD:", X_test_svd.shape)
print("Multiome latent:", Z_test_saved.shape)
print("Multiome cells:", multi_cell_ids.shape)
print("Multiome genes:", multi_gene_ids.shape)
print("Target SVD:", target_svd.components_.shape)

assert cite_predictions.shape == (48663, 140)
assert X_test_svd.shape == (55935, 128)
assert Z_test_saved.shape == (55935, 256)
assert multi_cell_ids.shape == (55935,)
assert multi_gene_ids.shape == (23418,)
assert target_svd.components_.shape == (256, 23418)

CITE predictions: (48663, 140)
Multiome input SVD: (55935, 128)
Multiome latent: (55935, 256)
Multiome cells: (55935,)
Multiome genes: (23418,)
Target SVD: (256, 23418)


In [12]:
PREVIEW_N = 256

X_preview = np.asarray(
    X_test_svd[:PREVIEW_N],
    dtype=np.float32
)

X_preview_scaled = (
    x_scaler
    .transform(X_preview)
    .astype(np.float32)
)

Z_preview_scaled = np.zeros(
    (PREVIEW_N, 256),
    dtype=np.float32
)

for model_path in mlp_paths:
    print("Checking:", model_path.name)

    model = tf.keras.models.load_model(
        model_path,
        compile=False
    )

    current_prediction = model.predict(
        X_preview_scaled,
        batch_size=256,
        verbose=0
    ).astype(np.float32)

    Z_preview_scaled += (
        current_prediction
        / len(mlp_paths)
    )

    del model, current_prediction
    gc.collect()


Z_preview_unscaled = (
    z_scaler
    .inverse_transform(
        Z_preview_scaled
    )
    .astype(np.float32)
)

Z_preview_saved = np.asarray(
    Z_test_saved[:PREVIEW_N],
    dtype=np.float32
)

scaled_mae = float(
    np.mean(
        np.abs(
            Z_preview_saved
            - Z_preview_scaled
        )
    )
)

unscaled_mae = float(
    np.mean(
        np.abs(
            Z_preview_saved
            - Z_preview_unscaled
        )
    )
)

scaled_max_error = float(
    np.max(
        np.abs(
            Z_preview_saved
            - Z_preview_scaled
        )
    )
)

unscaled_max_error = float(
    np.max(
        np.abs(
            Z_preview_saved
            - Z_preview_unscaled
        )
    )
)

print("\nLatent-space comparison")
print("=" * 60)
print("Scaled MAE:        ", scaled_mae)
print("Unscaled MAE:      ", unscaled_mae)
print("Scaled max error:  ", scaled_max_error)
print("Unscaled max error:", unscaled_max_error)

if scaled_mae < unscaled_mae:
    Z_TEST_IS_SCALED = True
    print(
        "\nConclusion: saved Z is SCALED."
    )
else:
    Z_TEST_IS_SCALED = False
    print(
        "\nConclusion: saved Z is already UNSCALED."
    )

Checking: final_mlp_seed_123.keras


2026-10-08 07:36:43.723060: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Checking: final_mlp_seed_2026.keras
Checking: final_mlp_seed_2027.keras
Checking: final_mlp_seed_42.keras
Checking: final_mlp_seed_7.keras

Latent-space comparison
Scaled MAE:         0.12616269290447235
Unscaled MAE:       1.1465178806346898e-10
Scaled max error:   5.650535583496094
Unscaled max error: 5.960464477539063e-08

Conclusion: saved Z is already UNSCALED.


In [13]:
Z_preview_for_inverse = np.asarray(
    Z_test_saved[:32],
    dtype=np.float32
)

if Z_TEST_IS_SCALED:
    Z_preview_for_inverse = (
        z_scaler
        .inverse_transform(
            Z_preview_for_inverse
        )
        .astype(np.float32)
    )

Y_multi_preview = (
    target_svd
    .inverse_transform(
        Z_preview_for_inverse
    )
    .astype(np.float32)
)

# 之前验证表明 clipping 略有提升
np.maximum(
    Y_multi_preview,
    0.0,
    out=Y_multi_preview
)

print("Preview shape:", Y_multi_preview.shape)
print("Minimum:", Y_multi_preview.min())
print("Maximum:", Y_multi_preview.max())
print("Mean:", Y_multi_preview.mean())
print("Standard deviation:", Y_multi_preview.std())
print("Finite:", np.isfinite(Y_multi_preview).all())

assert Y_multi_preview.shape == (
    32,
    23418
)

assert np.isfinite(
    Y_multi_preview
).all()

Preview shape: (32, 23418)
Minimum: 0.0
Maximum: 0.03536731
Mean: 0.0026916338
Standard deviation: 0.004072414
Finite: True


In [14]:
cite_cell_index = pd.Index(
    cite_predictions.index.astype(str)
)

cite_gene_index = pd.Index(
    cite_predictions.columns.astype(str)
)

cite_prediction_values = (
    cite_predictions
    .to_numpy(dtype=np.float32)
)

multi_cell_index = pd.Index(
    multi_cell_ids
)

multi_gene_index = pd.Index(
    multi_gene_ids
)

print("CITE cells:", len(cite_cell_index))
print("CITE proteins:", len(cite_gene_index))
print("Multiome cells:", len(multi_cell_index))
print("Multiome genes:", len(multi_gene_index))

# 两个任务的 test cells 不应重叠
overlap = cite_cell_index.intersection(
    multi_cell_index
)

print("Overlapping test cells:", len(overlap))

assert len(overlap) == 0

assert cite_cell_index.is_unique
assert cite_gene_index.is_unique
assert multi_cell_index.is_unique
assert multi_gene_index.is_unique

CITE cells: 48663
CITE proteins: 140
Multiome cells: 55935
Multiome genes: 23418
Overlapping test cells: 0


In [15]:
def get_multiome_targets(
    multi_row_indices,
    multi_column_indices,
    unique_cell_batch_size=256
):
    """
    根据 Multiome cell row 和 gene column，
    返回对应预测值。

    不生成完整的 55935 × 23418 矩阵。
    """

    multi_row_indices = np.asarray(
        multi_row_indices,
        dtype=np.int64
    )

    multi_column_indices = np.asarray(
        multi_column_indices,
        dtype=np.int64
    )

    if (
        multi_row_indices.shape
        != multi_column_indices.shape
    ):
        raise ValueError(
            "Multiome row/column shapes differ."
        )

    output = np.empty(
        len(multi_row_indices),
        dtype=np.float32
    )

    unique_rows, inverse = np.unique(
        multi_row_indices,
        return_inverse=True
    )

    # 将 evaluation pairs 按 unique cell 排序，
    # 便于分批恢复对应 cells
    pair_order = np.argsort(
        inverse,
        kind="stable"
    )

    inverse_sorted = inverse[pair_order]

    for unique_start in range(
        0,
        len(unique_rows),
        unique_cell_batch_size
    ):
        unique_stop = min(
            unique_start
            + unique_cell_batch_size,
            len(unique_rows)
        )

        current_rows = unique_rows[
            unique_start:unique_stop
        ]

        Z_batch = np.asarray(
            Z_test_saved[current_rows],
            dtype=np.float32
        )

        if Z_TEST_IS_SCALED:
            Z_batch = (
                z_scaler
                .inverse_transform(Z_batch)
                .astype(np.float32)
            )

        Y_batch = (
            target_svd
            .inverse_transform(Z_batch)
            .astype(np.float32)
        )

        # 使用验证中表现更好的 clipped prediction
        np.maximum(
            Y_batch,
            0.0,
            out=Y_batch
        )

        left = np.searchsorted(
            inverse_sorted,
            unique_start,
            side="left"
        )

        right = np.searchsorted(
            inverse_sorted,
            unique_stop,
            side="left"
        )

        current_pair_positions = (
            pair_order[left:right]
        )

        current_local_rows = (
            inverse[
                current_pair_positions
            ]
            - unique_start
        )

        current_columns = (
            multi_column_indices[
                current_pair_positions
            ]
        )

        output[
            current_pair_positions
        ] = Y_batch[
            current_local_rows,
            current_columns
        ]

        del Z_batch, Y_batch
        gc.collect()

    return output

In [16]:
def predict_evaluation_chunk(
    evaluation_chunk
):
    required_columns = {
        "row_id",
        "cell_id",
        "gene_id",
    }

    if not required_columns.issubset(
        evaluation_chunk.columns
    ):
        raise ValueError(
            "evaluation_ids.csv 缺少列："
            f"{required_columns}"
        )

    cell_ids = (
        evaluation_chunk["cell_id"]
        .astype(str)
        .to_numpy()
    )

    gene_ids = (
        evaluation_chunk["gene_id"]
        .astype(str)
        .to_numpy()
    )

    # 先尝试将每个 cell 映射到 CITE
    cite_rows_all = (
        cite_cell_index
        .get_indexer(cell_ids)
    )

    cite_mask = (
        cite_rows_all >= 0
    )

    multi_mask = ~cite_mask

    target_values = np.empty(
        len(evaluation_chunk),
        dtype=np.float32
    )

    # --------------------------------------------------
    # CITE
    # --------------------------------------------------

    if cite_mask.any():
        cite_rows = (
            cite_rows_all[cite_mask]
        )

        cite_columns = (
            cite_gene_index.get_indexer(
                gene_ids[cite_mask]
            )
        )

        if np.any(cite_columns < 0):
            unknown = np.unique(
                gene_ids[cite_mask][
                    cite_columns < 0
                ]
            )[:10]

            raise KeyError(
                "Unknown CITE protein IDs: "
                f"{unknown}"
            )

        target_values[cite_mask] = (
            cite_prediction_values[
                cite_rows,
                cite_columns
            ]
        )

    # --------------------------------------------------
    # Multiome
    # --------------------------------------------------

    if multi_mask.any():
        multi_rows = (
            multi_cell_index.get_indexer(
                cell_ids[multi_mask]
            )
        )

        multi_columns = (
            multi_gene_index.get_indexer(
                gene_ids[multi_mask]
            )
        )

        if np.any(multi_rows < 0):
            unknown = np.unique(
                cell_ids[multi_mask][
                    multi_rows < 0
                ]
            )[:10]

            raise KeyError(
                "Unknown test cell IDs: "
                f"{unknown}"
            )

        if np.any(multi_columns < 0):
            unknown = np.unique(
                gene_ids[multi_mask][
                    multi_columns < 0
                ]
            )[:10]

            raise KeyError(
                "Unknown Multiome gene IDs: "
                f"{unknown}"
            )

        target_values[multi_mask] = (
            get_multiome_targets(
                multi_rows,
                multi_columns,
                unique_cell_batch_size=256
            )
        )

    if not np.isfinite(
        target_values
    ).all():
        raise ValueError(
            "Generated target contains NaN/Inf."
        )

    return (
        target_values,
        int(cite_mask.sum()),
        int(multi_mask.sum()),
    )

In [17]:
DRY_RUN_ROWS = 200_000

evaluation_preview = pd.read_csv(
    files["evaluation_ids"],
    nrows=DRY_RUN_ROWS,
    dtype={
        "row_id": np.int64,
        "cell_id": str,
        "gene_id": str,
    }
)

print("Evaluation preview:")
display(evaluation_preview.head())

preview_targets, preview_cite_count, preview_multi_count = (
    predict_evaluation_chunk(
        evaluation_preview
    )
)

submission_preview = pd.DataFrame(
    {
        "row_id":
            evaluation_preview[
                "row_id"
            ].to_numpy(dtype=np.int64),

        "target":
            preview_targets,
    }
)

print("\nDry-run summary")
print("=" * 60)
print("Rows:", len(submission_preview))
print("CITE rows:", preview_cite_count)
print("Multiome rows:", preview_multi_count)
print("NaN:", submission_preview["target"].isna().any())
print("Minimum:", submission_preview["target"].min())
print("Maximum:", submission_preview["target"].max())

display(submission_preview.head())

assert len(submission_preview) == DRY_RUN_ROWS
assert submission_preview["row_id"].is_unique
assert np.isfinite(
    submission_preview["target"]
).all()

Evaluation preview:


,row_id,cell_id,gene_id
0,0,c2150f55becb,CD86
1,1,c2150f55becb,CD274
2,2,c2150f55becb,CD270
3,3,c2150f55becb,CD155
4,4,c2150f55becb,CD112



Dry-run summary
Rows: 200000
CITE rows: 200000
Multiome rows: 0
NaN: False
Minimum: -3.9985058307647705
Maximum: 38.410926818847656


,row_id,target
0,0,0.537193
1,1,0.402024
2,2,0.827317
3,3,4.379986
4,4,5.268987


In [18]:
sample_preview = pd.read_csv(
    files["sample_submission"],
    nrows=5
)

print("Official sample submission:")
display(sample_preview)

print(
    "Our columns:",
    submission_preview.columns.tolist()
)

print(
    "Official columns:",
    sample_preview.columns.tolist()
)

assert (
    submission_preview.columns.tolist()
    ==
    sample_preview.columns.tolist()
)

Official sample submission:


,row_id,target
0,0,0.0
1,1,0.0
2,2,0.0
3,3,0.0
4,4,0.0


Our columns: ['row_id', 'target']
Official columns: ['row_id', 'target']


In [19]:
PREVIEW_PATH = Path(
    "/kaggle/working/"
    "submission_preview.csv"
)

submission_preview.to_csv(
    PREVIEW_PATH,
    index=False,
    float_format="%.6g"
)

print("Saved:", PREVIEW_PATH)

Saved: /kaggle/working/submission_preview.csv


Final outcomes

In [21]:
EVALUATION_CHUNK_SIZE = 1_000_000

SUBMISSION_ZIP_PATH = Path(
    "/kaggle/working/"
    "open_problems_submission.zip"
)

INTERNAL_CSV_NAME = "submission.csv"

expected_cite_rows = (
    len(cite_cell_index)
    * len(cite_gene_index)
)

expected_multi_rows = (
    len(multi_cell_index)
    * len(multi_gene_index)
)

expected_total_rows = (
    expected_cite_rows
    + expected_multi_rows
)

print("Expected CITE rows:", f"{expected_cite_rows:,}")
print("Expected Multiome rows:", f"{expected_multi_rows:,}")
print("Expected total rows:", f"{expected_total_rows:,}")

Expected CITE rows: 6,812,820
Expected Multiome rows: 1,309,885,830
Expected total rows: 1,316,698,650


In [23]:
total_rows_written = 0
total_cite_rows = 0
total_multi_rows = 0
previous_last_row_id = None
first_output_chunk = True

estimated_chunks = math.ceil(
    expected_total_rows
    / EVALUATION_CHUNK_SIZE
)

evaluation_reader = pd.read_csv(
    files["evaluation_ids"],
    chunksize=EVALUATION_CHUNK_SIZE,
    dtype={
        "row_id": np.int64,
        "cell_id": str,
        "gene_id": str,
    }
)

with zipfile.ZipFile(
    SUBMISSION_ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=4,
    allowZip64=True,
) as archive:

    with archive.open(
        INTERNAL_CSV_NAME,
        mode="w",
        force_zip64=True,
    ) as binary_output:

        text_output = io.TextIOWrapper(
            binary_output,
            encoding="utf-8",
            newline="",
        )

        progress = tqdm(
            evaluation_reader,
            total=estimated_chunks,
            desc="Building submission",
        )

        for chunk_number, evaluation_chunk in enumerate(
            progress,
            start=1,
        ):
            row_ids = (
                evaluation_chunk["row_id"]
                .to_numpy(dtype=np.int64)
            )

            if len(row_ids) == 0:
                continue

            # 检查 evaluation_ids 的全局顺序
            if (
                previous_last_row_id is not None
                and row_ids[0]
                <= previous_last_row_id
            ):
                raise RuntimeError(
                    "evaluation_ids row_id "
                    "不是严格递增顺序。"
                )

            previous_last_row_id = int(
                row_ids[-1]
            )

            targets, cite_count, multi_count = (
                predict_evaluation_chunk(
                    evaluation_chunk
                )
            )

            output_chunk = pd.DataFrame(
                {
                    "row_id": row_ids,
                    "target": targets,
                }
            )

            output_chunk.to_csv(
                text_output,
                index=False,
                header=first_output_chunk,
                float_format="%.6g",
                lineterminator="\n",
            )

            first_output_chunk = False

            # 每个 chunk 都刷入压缩文件
            text_output.flush()

            total_rows_written += len(
                output_chunk
            )

            total_cite_rows += cite_count
            total_multi_rows += multi_count

            if chunk_number % 20 == 0:
                current_zip_size_gb = (
                    SUBMISSION_ZIP_PATH
                    .stat()
                    .st_size
                    / 1024**3
                )

                progress.set_postfix(
                    rows=f"{total_rows_written:,}",
                    zip_gb=f"{current_zip_size_gb:.2f}",
                )

            del (
                evaluation_chunk,
                output_chunk,
                targets,
                row_ids,
            )

            gc.collect()

        text_output.flush()

        # 避免 TextIOWrapper 在 binary_output 关闭后再次关闭底层对象
        text_output.detach()

Building submission:   0%|          | 0/1317 [00:00<?, ?it/s]

In [24]:
"target_svd" in globals(), "cite_predictions" in globals()

(True, True)

In [25]:
from pathlib import Path
import zipfile

print("=" * 70)
print("1. Possible limit variables")
print("=" * 70)

keywords = (
    "MAX", "LIMIT", "STOP", "DEBUG",
    "TEST", "CHUNK", "NROWS", "ROWS"
)

for name, value in sorted(globals().items()):
    if (
        any(word in name.upper() for word in keywords)
        and isinstance(value, (int, float, bool, str, type(None)))
    ):
        print(f"{name:<40} = {value!r}")


print("\n" + "=" * 70)
print("2. Suspicious lines in previous notebook cells")
print("=" * 70)

search_terms = (
    "60_000_000",
    "60000000",
    "MAX_ROWS",
    "MAX_CHUNKS",
    "STOP_AFTER",
    "nrows=",
    "break",
    "except",
    "Building submission"
)

for cell_number, source in enumerate(In):
    if not isinstance(source, str):
        continue

    matched_lines = []

    for line_number, line in enumerate(
        source.splitlines(),
        start=1
    ):
        if any(term in line for term in search_terms):
            matched_lines.append(
                f"    line {line_number}: {line.strip()}"
            )

    if matched_lines:
        print(f"\nCell In[{cell_number}]")
        print("\n".join(matched_lines))


print("\n" + "=" * 70)
print("3. Partial ZIP status")
print("=" * 70)

possible_outputs = [
    Path("/kaggle/working/open_problems_submission.zip"),
    Path("/kaggle/working/submission.zip"),
]

found_output = False

for zip_path in possible_outputs:
    if not zip_path.is_file():
        continue

    found_output = True

    print("\nFile:", zip_path)
    print(
        "ZIP size:",
        round(zip_path.stat().st_size / 1024**3, 3),
        "GB"
    )

    try:
        with zipfile.ZipFile(zip_path, "r") as archive:
            print("ZIP structure: valid")

            for info in archive.infolist():
                print("Member:", info.filename)
                print(
                    "Uncompressed size:",
                    round(info.file_size / 1024**3, 3),
                    "GB"
                )
                print(
                    "Compressed size:",
                    round(info.compress_size / 1024**3, 3),
                    "GB"
                )

    except zipfile.BadZipFile as error:
        print("ZIP structure: invalid/incomplete")
        print("Reason:", repr(error))

if not found_output:
    print("No expected output ZIP found.")

1. Possible limit variables
DRY_RUN_ROWS                             = 200000
EVALUATION_CHUNK_SIZE                    = 1000000
Z_TEST_IS_SCALED                         = False
chunk_number                             = 66
estimated_chunks                         = 1317
expected_cite_rows                       = 6812820
expected_multi_rows                      = 1309885830
expected_total_rows                      = 1316698650
first_output_chunk                       = False
scaled_max_error                         = 5.650535583496094
total_cite_rows                          = 6812820
total_multi_rows                         = 58931360
total_rows_written                       = 65744180
unscaled_max_error                       = 5.960464477539063e-08

2. Suspicious lines in previous notebook cells

Cell In[9]
    line 12: break

Cell In[17]
    line 5: nrows=DRY_RUN_ROWS,

Cell In[18]
    line 3: nrows=5

Cell In[22]
    line 45: desc="Building submission",

Cell In[23]
    line 45: de

In [26]:
from pathlib import Path
import zipfile

SUBMISSION_ZIP = Path(
    "/kaggle/working/open_problems_submission.zip"
)

EXPECTED_ROWS = 65_744_180

with zipfile.ZipFile(SUBMISSION_ZIP, "r") as archive:
    assert "submission.csv" in archive.namelist()

    line_count = 0
    first_line = None
    last_line = None

    with archive.open("submission.csv", "r") as file:
        while True:
            block = file.read(16 * 1024 * 1024)

            if not block:
                break

            if first_line is None:
                first_line = block.split(b"\n", 1)[0]

            line_count += block.count(b"\n")

            nonempty_lines = [
                line for line in block.split(b"\n")
                if line
            ]

            if nonempty_lines:
                last_line = nonempty_lines[-1]

data_rows = line_count - 1

print("Header:", first_line.decode())
print("Last line:", last_line.decode())
print("CSV data rows:", f"{data_rows:,}")
print("Expected rows:", f"{EXPECTED_ROWS:,}")
print(
    "ZIP size:",
    f"{SUBMISSION_ZIP.stat().st_size / 1024**3:.3f} GB"
)

assert first_line.decode().strip() == "row_id,target"
assert data_rows == EXPECTED_ROWS

print("\nFinal submission validation passed.")
print("Ready to submit:", SUBMISSION_ZIP)

Header: row_id,target
Last line: 65744179,0.0176073
CSV data rows: 65,744,180
Expected rows: 65,744,180
ZIP size: 0.409 GB

Final submission validation passed.
Ready to submit: /kaggle/working/open_problems_submission.zip


In [28]:
from IPython.display import FileLink, display

display(
    FileLink(
        "/kaggle/working/open_problems_submission.zip"
    )
)

/kaggle/working/open_problems_submission.zip